# Gold Layer – disease_medication_pharmacy_insurance_gold

## Purpose
This notebook creates an enriched Gold analytical table by combining the main Gold healthcare dataset with insurance coverage information.

## Business Goal
The purpose of this table is to extend medication availability analysis with insurance-related insights, such as:
- which medications are covered by insurance
- which insurance plans are linked to available medications
- how insurance information can support business reporting and dashboard analysis

## Source Tables
- `capstone.gold.disease_medication_pharmacy_gold`
- `capstone.silver.insurance`

## Output Table
- `capstone.gold.disease_medication_pharmacy_insurance_gold`

## Grain
One row represents:

**one disease + one medication + one pharmacy + one insurance plan combination**

## Notes
This table is designed as an enriched analytical dataset for optional reporting and dashboard extensions related to medication coverage and insurance information.

In [0]:
%sql
DESCRIBE capstone.silver.insurance;

In [0]:
%sql
SELECT *
FROM capstone.silver.insurance
LIMIT 20;

In [0]:
%sql
-- ============================================================
-- Gold Table: disease_medication_pharmacy_insurance_gold
-- Purpose:
-- Create an enriched Gold dataset that links diseases,
-- medications, pharmacies, inventory availability,
-- and insurance coverage information.
--
-- Target table:
-- capstone.gold.disease_medication_pharmacy_insurance_gold
-- ============================================================

CREATE OR REPLACE TABLE capstone.gold.disease_medication_pharmacy_insurance_gold
USING DELTA
AS

WITH

main_gold_base AS (
    SELECT DISTINCT
        disease_id,
        disease_name,
        medication_id,
        medication_name,
        pharmacy_id,
        pharmacy_name,
        city,
        county,
        state,
        region,
        price,
        quantity,
        last_updated
    FROM capstone.gold.disease_medication_pharmacy_gold
),

insurance_base AS (
    SELECT
        insurance_id,
        insurance_name,
        plan_type,
        medication_id_reference,
        medication_name_reference,
        disease_name_reference,
        copay,
        coinsurance,
        requires_prior_auth,
        requires_step_therapy,
        is_specialty_drug,
        coverage_status,
        source_system,
        source_record_id
    FROM (
        SELECT
            insurance_id,
            insurance_name,
            plan_type,
            medication_id_reference,
            medication_name_reference,
            disease_name_reference,
            copay,
            coinsurance,
            requires_prior_auth,
            requires_step_therapy,
            is_specialty_drug,
            coverage_status,
            source_system,
            source_record_id,
            ROW_NUMBER() OVER (
                PARTITION BY LOWER(TRIM(medication_name_reference))
                ORDER BY insurance_id
            ) AS rn
        FROM capstone.silver.insurance
        WHERE insurance_id IS NOT NULL
          AND TRIM(insurance_id) <> ''
          AND medication_name_reference IS NOT NULL
          AND TRIM(medication_name_reference) <> ''
    ) i
    WHERE rn = 1
),

gold_dataset AS (
    SELECT
        g.disease_id,
        g.disease_name,
        g.medication_id,
        g.medication_name,
        g.pharmacy_id,
        g.pharmacy_name,
        g.city,
        g.county,
        g.state,
        g.region,
        g.price,
        g.quantity,
        g.last_updated,
        i.insurance_id,
        i.insurance_name,
        i.plan_type,
        i.medication_id_reference,
        i.medication_name_reference,
        i.disease_name_reference,
        i.copay,
        i.coinsurance,
        i.requires_prior_auth,
        i.requires_step_therapy,
        i.is_specialty_drug,
        i.coverage_status,
        i.source_system AS insurance_source_system,
        i.source_record_id AS insurance_source_record_id
    FROM main_gold_base g
    LEFT JOIN insurance_base i
        ON LOWER(TRIM(g.medication_name)) = LOWER(TRIM(i.medication_name_reference))
)

SELECT DISTINCT *
FROM gold_dataset;

In [0]:
%sql
SELECT 'Total Rows' AS check_name, COUNT(*) AS result
FROM capstone.gold.disease_medication_pharmacy_insurance_gold

UNION ALL

SELECT 'Rows Without Insurance Match', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_insurance_gold
WHERE insurance_id IS NULL OR TRIM(insurance_id) = ''

UNION ALL

SELECT 'Rows Without Insurance Name', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_insurance_gold
WHERE insurance_name IS NULL OR TRIM(insurance_name) = ''

UNION ALL

SELECT 'Rows Without Coverage Status', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_insurance_gold
WHERE coverage_status IS NULL OR TRIM(coverage_status) = ''

UNION ALL

SELECT 'Missing Medication ID', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_insurance_gold
WHERE medication_id IS NULL OR TRIM(medication_id) = ''

UNION ALL

SELECT 'Missing Pharmacy ID', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_insurance_gold
WHERE pharmacy_id IS NULL OR TRIM(pharmacy_id) = ''

UNION ALL

SELECT 'Negative Copay', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_insurance_gold
WHERE copay < 0

UNION ALL

SELECT 'Negative Coinsurance', COUNT(*)
FROM capstone.gold.disease_medication_pharmacy_insurance_gold
WHERE coinsurance < 0

UNION ALL

SELECT 'Duplicate Business Rows', COUNT(*)
FROM (
    SELECT
        disease_id,
        medication_id,
        pharmacy_id,
        insurance_id,
        COUNT(*) AS cnt
    FROM capstone.gold.disease_medication_pharmacy_insurance_gold
    GROUP BY disease_id, medication_id, pharmacy_id, insurance_id
    HAVING COUNT(*) > 1
) dup;

In [0]:
%sql
SELECT
    disease_name,
    medication_name,
    pharmacy_name,
    state,
    region,
    insurance_name,
    plan_type,
    coverage_status,
    copay,
    coinsurance
FROM capstone.gold.disease_medication_pharmacy_insurance_gold
LIMIT 20;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT disease_id) AS total_diseases,
    COUNT(DISTINCT medication_id) AS total_medications,
    COUNT(DISTINCT pharmacy_id) AS total_pharmacies,
    COUNT(DISTINCT insurance_id) AS total_insurance_plans
FROM capstone.gold.disease_medication_pharmacy_insurance_gold;

In [0]:
%sql
SELECT
    insurance_name,
    coverage_status,
    COUNT(*) AS total_records
FROM capstone.gold.disease_medication_pharmacy_insurance_gold
WHERE insurance_id IS NOT NULL
GROUP BY insurance_name, coverage_status
ORDER BY total_records DESC;

In [0]:
%sql
CREATE OR REPLACE VIEW capstone.kpi.kpi_prior_authorization_by_disease AS
SELECT
    disease_id,
    disease_name,
    CASE
        WHEN requires_prior_auth = true THEN 'Prior Authorization Required'
        WHEN requires_prior_auth = false THEN 'No Prior Authorization'
        ELSE 'Unknown'
    END AS prior_auth_status,
    COUNT(*) AS total_records,
    COUNT(DISTINCT medication_id) AS medication_count,
    COUNT(DISTINCT insurance_id) AS insurance_plan_count
FROM capstone.gold.disease_medication_pharmacy_insurance_gold
WHERE insurance_id IS NOT NULL
GROUP BY
    disease_id,
    disease_name,
    CASE
        WHEN requires_prior_auth = true THEN 'Prior Authorization Required'
        WHEN requires_prior_auth = false THEN 'No Prior Authorization'
        ELSE 'Unknown'
    END;

In [0]:
%sql
SELECT *
FROM capstone.kpi.kpi_insurance_coverage_by_disease
ORDER BY total_records DESC
LIMIT 20;